# Multi-Resolution Attention Audio Classification on ESC-50

**Deep Learning & Signal Processing Benchmark**

Comparing a **Multi-Resolution Attention Network** (`MultiResAttentionNet`) against a matched **Single-Resolution CNN Baseline** (`SingleResCNN`) on the official ESC-50 dataset.

### Instructions for Google Colab:
1. Go to `Runtime` > `Change runtime type` > Select **T4 GPU**.
2. Run all cells sequentially.

In [ ]:
# Install dependencies
!pip install torch torchaudio torchvision scikit-learn pandas matplotlib soundfile -q

In [ ]:
# Clone ESC-50 dataset
import os
if not os.path.exists('ESC-50'):
    !git clone --depth 1 https://github.com/karolpiczak/ESC-50.git
!mkdir -p results

## 1. Data Pipeline (`data.py`)

In [ ]:
%%writefile data.py
import os
import pandas as pd
import torch
import torchaudio
from torch.utils.data import Dataset

SAMPLE_RATE = 44100
N_MELS = 64
FIXED_FRAMES = 216

RESOLUTIONS = {
    "fine":   dict(n_fft=1024, hop_length=128),
    "mid":    dict(n_fft=1024, hop_length=512),
    "coarse": dict(n_fft=2048, hop_length=1024),
}

class MultiResMelExtractor:
    def __init__(self, sample_rate=SAMPLE_RATE, n_mels=N_MELS, fixed_frames=FIXED_FRAMES, augment=False):
        self.sample_rate = sample_rate
        self.n_mels = n_mels
        self.fixed_frames = fixed_frames
        self.augment = augment
        self.transforms = {
            name: torchaudio.transforms.MelSpectrogram(
                sample_rate=sample_rate, n_fft=cfg["n_fft"], hop_length=cfg["hop_length"], n_mels=n_mels
            )
            for name, cfg in RESOLUTIONS.items()
        }
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=12)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=24)

    def _process_one(self, wav, name):
        mel = self.transforms[name](wav)
        mel = self.db(mel)
        mel = torch.nn.functional.interpolate(
            mel.unsqueeze(0), size=(self.n_mels, self.fixed_frames), mode="bilinear", align_corners=False
        ).squeeze(0)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        if self.augment:
            mel = self.freq_mask(mel)
            mel = self.time_mask(mel)
        return mel

    def __call__(self, wav):
        feats = [self._process_one(wav, name) for name in ["fine", "mid", "coarse"]]
        return torch.cat(feats, dim=0)

def load_audio(path):
    try:
        wav, sr = torchaudio.load(path, backend="soundfile")
        return wav, sr
    except Exception:
        pass
    try:
        import soundfile as sf
        data, sr = sf.read(path, dtype="float32")
        wav = torch.from_numpy(data)
        if wav.ndim == 1:
            wav = wav.unsqueeze(0)
        else:
            wav = wav.transpose(0, 1)
        return wav, sr
    except Exception:
        import numpy as np
        import scipy.io.wavfile as wavfile
        sr, data = wavfile.read(path)
        wav = torch.from_numpy(data).float()
        if data.dtype == np.int16:
            wav = wav / 32768.0
        elif data.dtype == np.int32:
            wav = wav / 2147483648.0
        if wav.ndim == 1:
            wav = wav.unsqueeze(0)
        else:
            wav = wav.transpose(0, 1)
        return wav, sr

FEATURE_CACHE_PATH = os.path.join("results", "esc50_features.pt")
_GLOBAL_FEATURE_CACHE = None

def precompute_and_cache_features(root_dir, cache_path=FEATURE_CACHE_PATH):
    global _GLOBAL_FEATURE_CACHE
    if _GLOBAL_FEATURE_CACHE is not None:
        return _GLOBAL_FEATURE_CACHE
    if os.path.exists(cache_path):
        print(f"Loading precomputed features from {cache_path}...")
        _GLOBAL_FEATURE_CACHE = torch.load(cache_path)
        return _GLOBAL_FEATURE_CACHE

    print("Precomputing multi-resolution mel-spectrograms for ESC-50...")
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    meta = pd.read_csv(os.path.join(root_dir, "meta", "esc50.csv"))
    audio_dir = os.path.join(root_dir, "audio")
    clean_extractor = MultiResMelExtractor(sample_rate=SAMPLE_RATE, augment=False)
    cache = {}
    for idx in range(len(meta)):
        row = meta.iloc[idx]
        path = os.path.join(audio_dir, row.filename)
        wav, sr = load_audio(path)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        feat = clean_extractor(wav)
        cache[row.filename] = feat.cpu()
    torch.save(cache, cache_path)
    _GLOBAL_FEATURE_CACHE = cache
    return _GLOBAL_FEATURE_CACHE

class ESC50Dataset(Dataset):
    def __init__(self, root_dir, folds, extractor=None, sample_rate=SAMPLE_RATE):
        self.root_dir = root_dir
        meta = pd.read_csv(os.path.join(root_dir, "meta", "esc50.csv"))
        self.meta = meta[meta.fold.isin(folds)].reset_index(drop=True)
        self.sample_rate = sample_rate
        self.extractor = extractor or MultiResMelExtractor(sample_rate=sample_rate)
        self.classes = sorted(meta.category.unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.cache = precompute_and_cache_features(root_dir)

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        feat = self.cache[row.filename]
        if self.extractor.augment:
            feat = feat.clone()
            feat = self.extractor.freq_mask(feat)
            feat = self.extractor.time_mask(feat)
            if torch.rand(1).item() > 0.5:
                shift = torch.randint(-18, 18, (1,)).item()
                feat = torch.roll(feat, shifts=shift, dims=-1)
        label = self.class_to_idx[row.category]
        return feat, label

def get_dataloaders(root_dir, batch_size=32, test_fold=5, val_fold=4, num_workers=2):
    from torch.utils.data import DataLoader
    train_folds = [f for f in [1, 2, 3, 4, 5] if f not in (test_fold, val_fold)]
    train_extractor = MultiResMelExtractor(augment=True)
    eval_extractor = MultiResMelExtractor(augment=False)
    train_ds = ESC50Dataset(root_dir, train_folds, train_extractor)
    val_ds = ESC50Dataset(root_dir, [val_fold], eval_extractor)
    test_ds = ESC50Dataset(root_dir, [test_fold], eval_extractor)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers),
        train_ds.classes,
    )


## 2. Model Architectures (`models.py`)

In [ ]:
%%writefile models.py
import torch
import torch.nn as nn
import torchvision.models as tvm

def make_resnet_backbone(pretrained=True, freeze_early=True):
    weights = tvm.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    net = tvm.resnet18(weights=weights)
    old_conv = net.conv1
    new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                          stride=old_conv.stride, padding=old_conv.padding, bias=False)
    if pretrained:
        with torch.no_grad():
            new_conv.weight[:] = old_conv.weight.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    feat_dim = net.fc.in_features
    net.fc = nn.Identity()
    if freeze_early:
        for name, param in net.named_parameters():
            if name.startswith(("layer1", "layer2", "conv1", "bn1")):
                param.requires_grad = False
    return net, feat_dim

class SingleResCNN(nn.Module):
    def __init__(self, n_classes=50, pretrained=True):
        super().__init__()
        self.backbone, feat_dim = make_resnet_backbone(pretrained=pretrained)
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(feat_dim, n_classes),
        )
    def forward(self, x):
        if x.shape[1] == 3:
            x = x[:, 1:2, :, :]
        feat = self.backbone(x)
        return self.classifier(feat)

class TimeFreqAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.query = nn.Conv2d(channels, max(channels // reduction, 1), 1)
        self.key = nn.Conv2d(channels, max(channels // reduction, 1), 1)
        self.value = nn.Conv2d(channels, channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        B, C, H, W = x.shape
        q = self.query(x).view(B, -1, H * W).permute(0, 2, 1)
        k = self.key(x).view(B, -1, H * W)
        attn = torch.softmax(torch.bmm(q, k), dim=-1)
        v = self.value(x).view(B, C, H * W)
        out = torch.bmm(v, attn.permute(0, 2, 1)).view(B, C, H, W)
        return x + self.gamma * out

class ResBranch(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        backbone, feat_dim = make_resnet_backbone(pretrained=pretrained)
        self.stem = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1, backbone.layer2, backbone.layer3, backbone.layer4,
        )
        self.attn = TimeFreqAttention(feat_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = feat_dim
    def forward(self, x):
        f = self.stem(x)
        f = self.attn(f)
        return self.pool(f).flatten(1)

class MultiResAttentionNet(nn.Module):
    def __init__(self, n_classes=50, pretrained=True):
        super().__init__()
        self.branches = nn.ModuleDict({
            name: ResBranch(pretrained=pretrained) for name in ["fine", "mid", "coarse"]
        })
        branch_dim = self.branches["fine"].out_dim
        self.fusion_gate = nn.Sequential(
            nn.Linear(branch_dim * 3, 3),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(branch_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )
    def forward(self, x):
        fine = self.branches["fine"](x[:, 0:1])
        mid = self.branches["mid"](x[:, 1:2])
        coarse = self.branches["coarse"](x[:, 2:3])
        concat = torch.cat([fine, mid, coarse], dim=1)
        weights = torch.softmax(self.fusion_gate(concat), dim=1)
        stacked = torch.stack([fine, mid, coarse], dim=1)
        fused = (stacked * weights.unsqueeze(-1)).sum(dim=1)
        return self.classifier(fused)
    def get_fusion_weights(self, x):
        with torch.no_grad():
            fine = self.branches["fine"](x[:, 0:1])
            mid = self.branches["mid"](x[:, 1:2])
            coarse = self.branches["coarse"](x[:, 2:3])
            concat = torch.cat([fine, mid, coarse], dim=1)
            weights = torch.softmax(self.fusion_gate(concat), dim=1)
        return weights


## 3. Training Script (`train.py`)

In [ ]:
%%writefile train.py
import argparse, json, os, time, random
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, accuracy_score
from data import get_dataloaders
from models import MultiResAttentionNet, SingleResCNN

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def mixup(x, y, n_classes, alpha=0.3):
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1-lam) * x[idx]
    y_oh = torch.zeros(x.size(0), n_classes, device=x.device).scatter_(1, y.unsqueeze(1), 1)
    mixed_y = lam * y_oh + (1-lam) * y_oh[idx]
    return mixed_x, mixed_y

def run_epoch(model, loader, criterion, optimizer, device, train=True, n_classes=50, use_mixup=False):
    if train:
        model.train()
    else:
        model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train:
                optimizer.zero_grad()
                if use_mixup:
                    mx, my = mixup(x, y, n_classes)
                    out = model(mx)
                    logp = torch.log_softmax(out, dim=1)
                    loss = -(my * logp).sum(dim=1).mean()
                    preds, targets = out.argmax(1).cpu().tolist(), my.argmax(1).cpu().tolist()
                else:
                    out = model(x)
                    loss = criterion(out, y)
                    preds, targets = out.argmax(1).cpu().tolist(), y.cpu().tolist()
                loss.backward()
                optimizer.step()
            else:
                out = model(x)
                loss = criterion(out, y)
                preds, targets = out.argmax(1).cpu().tolist(), y.cpu().tolist()
            total_loss += loss.item() * x.size(0)
            all_preds += preds
            all_labels += targets
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / len(loader.dataset), acc, f1

def measure_latency(model, sample_input, device, n_runs=30):
    model.eval()
    sample_input = sample_input.to(device)
    bs = sample_input.size(0)
    with torch.no_grad():
        for _ in range(5):
            model(sample_input)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(n_runs):
            model(sample_input)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t1 = time.time()
    return ((t1 - t0) / n_runs * 1000) / bs

def train_one_split(args, device, test_fold, val_fold):
    train_loader, val_loader, test_loader, classes = get_dataloaders(
        args.data_root, batch_size=args.batch_size, test_fold=test_fold, val_fold=val_fold)
    n_classes = len(classes)
    model = (MultiResAttentionNet(n_classes, pretrained=not args.no_pretrained) if args.model == 'multires'
             else SingleResCNN(n_classes, pretrained=not args.no_pretrained)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    best_val_acc, best_state = 0.0, None
    for epoch in range(1, args.epochs + 1):
        train_loss, train_acc, _ = run_epoch(model, train_loader, criterion, optimizer, device, train=True, n_classes=n_classes, use_mixup=args.mixup)
        val_loss, val_acc, val_f1 = run_epoch(model, val_loader, criterion, optimizer, device, train=False, n_classes=n_classes)
        scheduler.step()
        print(f"[{args.model} fold(test={test_fold})] epoch {epoch:02d}/{args.epochs:02d} train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}", flush=True)
        if val_acc > best_val_acc or best_state is None:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    test_loss, test_acc, test_f1 = run_epoch(model, test_loader, criterion, optimizer, device, train=False, n_classes=n_classes)
    sample_x, _ = next(iter(test_loader))
    return {
        'test_fold': test_fold, 'test_accuracy': test_acc, 'test_macro_f1': test_f1,
        'best_val_accuracy': best_val_acc, 'num_params': sum(p.numel() for p in model.parameters()),
        'latency_ms_per_sample': measure_latency(model, sample_x, device),
    }, best_state

def main():
    p = argparse.ArgumentParser()
    p.add_argument('--data_root', required=True)
    p.add_argument('--model', choices=['multires', 'baseline'], default='multires')
    p.add_argument('--epochs', type=int, default=30)
    p.add_argument('--batch_size', type=int, default=32)
    p.add_argument('--lr', type=float, default=5e-4)
    p.add_argument('--weight_decay', type=float, default=5e-4)
    p.add_argument('--mixup', action='store_true')
    p.add_argument('--no_pretrained', action='store_true')
    p.add_argument('--cv', action='store_true')
    p.add_argument('--seed', type=int, default=42)
    args = p.parse_args()
    set_seed(args.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs('results', exist_ok=True)
    if args.cv:
        fold_results = []
        for test_fold in [1, 2, 3, 4, 5]:
            val_fold = test_fold - 1 if test_fold > 1 else 5
            metrics, best_state = train_one_split(args, device, test_fold, val_fold)
            fold_results.append(metrics)
            torch.save(best_state, f'results/{args.model}_fold{test_fold}_best.pt')
        import statistics
        accs, f1s = [r['test_accuracy'] for r in fold_results], [r['test_macro_f1'] for r in fold_results]
        summary = {
            'model': args.model, 'per_fold': fold_results,
            'mean_test_accuracy': statistics.mean(accs), 'std_test_accuracy': statistics.stdev(accs),
            'mean_test_macro_f1': statistics.mean(f1s), 'std_test_macro_f1': statistics.stdev(f1s),
            'num_params': fold_results[0]['num_params'],
            'latency_ms_per_sample': fold_results[0]['latency_ms_per_sample'], 'epochs': args.epochs,
        }
        with open(f'results/{args.model}_metrics.json', 'w') as f:
            json.dump(summary, f, indent=2)
    else:
        metrics, best_state = train_one_split(args, device, test_fold=5, val_fold=4)
        metrics['model'], metrics['epochs'] = args.model, args.epochs
        with open(f'results/{args.model}_metrics.json', 'w') as f:
            json.dump(metrics, f, indent=2)
        torch.save(best_state, f'results/{args.model}_best.pt')
        print(f"\nFinal test accuracy: {metrics['test_accuracy']:.4f} | macro-F1: {metrics['test_macro_f1']:.4f}")

if __name__ == '__main__':
    main()


## 4. Run Training

In [ ]:
# Train Baseline SingleResCNN
!python train.py --data_root ESC-50 --model baseline --epochs 30 --mixup

In [ ]:
# Train Proposed MultiResAttentionNet
!python train.py --data_root ESC-50 --model multires --epochs 30 --mixup

## 5. Compare Results & Generate 4-Panel Figure

In [ ]:
%%writefile compare_results.py
import json, os, matplotlib.pyplot as plt

with open("results/multires_metrics.json") as f: multires = json.load(f)
with open("results/baseline_metrics.json") as f: baseline = json.load(f)
is_cv = "mean_test_accuracy" in multires and "mean_test_accuracy" in baseline

print("\n" + "=" * 80)
print(f"{'EVALUATION BENCHMARK: BASELINE vs PROPOSED (ESC-50)':^80}")
print("=" * 80)
print(f"{'Metric':<30}{'Baseline (SingleRes CNN)':<25}{'Proposed (MultiRes+Attn)':<25}")
print("-" * 80)

b_acc = baseline.get('mean_test_accuracy', baseline.get('test_accuracy'))
m_acc = multires.get('mean_test_accuracy', multires.get('test_accuracy'))
b_f1 = baseline.get('mean_test_macro_f1', baseline.get('test_macro_f1'))
m_f1 = multires.get('mean_test_macro_f1', multires.get('test_macro_f1'))
b_err = baseline.get('std_test_accuracy', 0.0) if is_cv else 0.0
m_err = multires.get('std_test_accuracy', 0.0) if is_cv else 0.0

print(f"{'Test Accuracy':<30}{b_acc*100:.2f}%{' ± ' + str(round(b_err*100,2))+'%' if is_cv else '':<20}{m_acc*100:.2f}%{' ± ' + str(round(m_err*100,2))+'%' if is_cv else ''}")
print(f"{'Macro F1 Score':<30}{b_f1:.4f:<25}{m_f1:.4f:<25}")
print(f"{'Parameter Count':<30}{baseline['num_params']/1e6:.2f} M{'':<20}{multires['num_params']/1e6:.2f} M")
print(f"{'Batched Latency':<30}{baseline['latency_ms_per_sample']:.2f} ms/sample{'':<10}{multires['latency_ms_per_sample']:.2f} ms/sample")
print("=" * 80)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
labels, colors = ["Baseline", "Multi-Res + Attn"], ["#718096", "#2b6cb0"]
axes[0].bar(labels, [b_acc * 100, m_acc * 100], color=colors, yerr=[b_err*100, m_err*100], capsize=5)
axes[0].set_title("Test Accuracy (%)", fontweight="bold"); axes[0].set_ylim(0, 100)
axes[1].bar(labels, [b_f1, m_f1], color=colors)
axes[1].set_title("Macro F1 Score", fontweight="bold"); axes[1].set_ylim(0, 1.0)
axes[2].bar(labels, [baseline['num_params']/1e6, multires['num_params']/1e6], color=colors)
axes[2].set_title("Parameters (M)", fontweight="bold")
axes[3].bar(labels, [baseline['latency_ms_per_sample'], multires['latency_ms_per_sample']], color=colors)
axes[3].set_title("Latency (ms / sample)", fontweight="bold")
plt.tight_layout()
plt.savefig("results/comparison.png", dpi=200)
print("Saved chart to results/comparison.png")


In [ ]:
!python compare_results.py
from IPython.display import Image, display
display(Image('results/comparison.png'))

## 6. Single Audio Inference (`predict.py`)

In [ ]:
!python src/predict.py ESC-50/audio/1-100032-A-0.wav --model baseline --checkpoint results/baseline_best.pt
!python src/predict.py ESC-50/audio/1-100032-A-0.wav --model multires --checkpoint results/multires_best.pt